In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers, models
import pandas as pd


2025-10-09 21:24:26.983023: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-09 21:24:27.223209: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-09 21:24:28.555573: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-09 21:24:28.555573: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df = pd.read_csv("./dataset/results.csv")

In [3]:
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
48527,2025-09-09,Bosnia and Herzegovina,Austria,1,2,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,False
48528,2025-09-09,Cyprus,Romania,2,2,FIFA World Cup qualification,Nicosia,Cyprus,False
48529,2025-09-09,Norway,Moldova,11,1,FIFA World Cup qualification,Oslo,Norway,False
48530,2025-09-09,Albania,Latvia,1,0,FIFA World Cup qualification,Tirana,Albania,False


In [4]:
# make new column with win/loss/draw by compare the home_score and away_score
df['result'] = np.where(df['home_score'] > df['away_score'], 'win',
                        np.where(df['home_score'] < df['away_score'], 'loss', 'draw'))

In [5]:
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,draw
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,win
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,win
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,draw
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,win
...,...,...,...,...,...,...,...,...,...,...
48527,2025-09-09,Bosnia and Herzegovina,Austria,1,2,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,False,loss
48528,2025-09-09,Cyprus,Romania,2,2,FIFA World Cup qualification,Nicosia,Cyprus,False,draw
48529,2025-09-09,Norway,Moldova,11,1,FIFA World Cup qualification,Oslo,Norway,False,win
48530,2025-09-09,Albania,Latvia,1,0,FIFA World Cup qualification,Tirana,Albania,False,win


In [6]:
df.drop(columns=['home_score', 'away_score'], inplace=True)

In [7]:
df.drop(columns=["date"],inplace=True)

In [8]:
df

,home_team,away_team,tournament,city,country,neutral,result
0,Scotland,England,Friendly,Glasgow,Scotland,False,draw
1,England,Scotland,Friendly,London,England,False,win
2,Scotland,England,Friendly,Glasgow,Scotland,False,win
3,England,Scotland,Friendly,London,England,False,draw
4,Scotland,England,Friendly,Glasgow,Scotland,False,win
...,...,...,...,...,...,...,...
48527,Bosnia and Herzegovina,Austria,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,False,loss
48528,Cyprus,Romania,FIFA World Cup qualification,Nicosia,Cyprus,False,draw
48529,Norway,Moldova,FIFA World Cup qualification,Oslo,Norway,False,win
48530,Albania,Latvia,FIFA World Cup qualification,Tirana,Albania,False,win


In [9]:
X = df[['home_team', 'away_team', 'tournament', 'city', 'country', 'neutral']]
y = df['result']


In [10]:
from sklearn.preprocessing import LabelEncoder

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)  # win=2, draw=1, loss=0 (for example)

# One-hot encode categorical features
X = pd.get_dummies(X, drop_first=True)


In [11]:
df.isnull().sum()

home_team     0
away_team     0
tournament    0
city          0
country       0
neutral       0
result        0
dtype: int64

In [12]:
X

,neutral,home_team_Afghanistan,home_team_Albania,home_team_Alderney,home_team_Algeria,home_team_American Samoa,home_team_Andalusia,home_team_Andorra,home_team_Angola,home_team_Anguilla,...,country_Western Samoa,country_Yemen,country_Yemen AR,country_Yemen DPR,country_Yugoslavia,country_Zambia,country_Zanzibar,country_Zaïre,country_Zimbabwe,country_Éire
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48527,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
48528,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
48529,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
48530,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [13]:
from sklearn.model_selection import train_test_split
# Use stratify to preserve class distribution in train/test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [15]:
# Improved model: larger capacity, BatchNorm and Dropout to regularize
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(32),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.1),
    layers.Dense(3, activation='softmax')  # 3 classes: win, loss, draw
])


E0000 00:00:1760024377.633123  112700 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1760024377.723635  112700 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [16]:
# Compile with a tuned learning rate
opt = keras.optimizers.Adam(learning_rate=1e-3)
model.compile(optimizer=opt,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [19]:

# Callbacks: early stopping and LR reduction
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

# Train longer with a reasonable batch size
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, 
                    callbacks=callbacks)


Epoch 1/50


2025-10-09 21:25:31.719704: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 398810400 exceeds 10% of free system memory.


971/971 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.4757 - loss: 1.0706 - val_accuracy: 0.5315 - val_loss: 0.9914 - learning_rate: 0.0010
Epoch 2/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.4757 - loss: 1.0706 - val_accuracy: 0.5315 - val_loss: 0.9914 - learning_rate: 0.0010
Epoch 2/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5359 - loss: 0.9749 - val_accuracy: 0.5396 - val_loss: 0.9701 - learning_rate: 0.0010
Epoch 3/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5359 - loss: 0.9749 - val_accuracy: 0.5396 - val_loss: 0.9701 - learning_rate: 0.0010
Epoch 3/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5602 - loss: 0.9384 - val_accuracy: 0.5482 - val_loss: 0.9655 - learning_rate: 0.0010
Epoch 4/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5602 - loss: 0.9384 - val_accuracy: 0.5482 - val_loss: 0.9655 - learning_rate: 0.0010
Epoch 4/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5768 - loss: 0.9094 - val_accurac

In [20]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.3f}")


174/304 ━━━━━━━━━━━━━━━━━━━━ 0s 879us/step - accuracy: 0.5541 - loss: 0.9575

2025-10-09 21:26:58.484913: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 124637880 exceeds 10% of free system memory.


304/304 ━━━━━━━━━━━━━━━━━━━━ 0s 965us/step - accuracy: 0.5542 - loss: 0.9571
Test Accuracy: 0.554
304/304 ━━━━━━━━━━━━━━━━━━━━ 0s 965us/step - accuracy: 0.5542 - loss: 0.9571
Test Accuracy: 0.554


In [21]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

print(classification_report(y_test, y_pred, target_names=le.classes_))


154/304 ━━━━━━━━━━━━━━━━━━━━ 0s 657us/step

2025-10-09 21:26:59.990626: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 124637880 exceeds 10% of free system memory.


304/304 ━━━━━━━━━━━━━━━━━━━━ 0s 845us/step
304/304 ━━━━━━━━━━━━━━━━━━━━ 0s 845us/step
              precision    recall  f1-score   support

        draw       0.25      0.01      0.01      2207
        loss       0.49      0.49      0.49      2740
         win       0.58      0.85      0.69      4760

    accuracy                           0.55      9707
   macro avg       0.44      0.45      0.40      9707
weighted avg       0.48      0.55      0.48      9707

              precision    recall  f1-score   support

        draw       0.25      0.01      0.01      2207
        loss       0.49      0.49      0.49      2740
         win       0.58      0.85      0.69      4760

    accuracy                           0.55      9707
   macro avg       0.44      0.45      0.40      9707
weighted avg       0.48      0.55      0.48      9707

